### Import des packages

In [ ]:
# Importer les packages
import matplotlib.pyplot as plt
import seaborn as sns
import keras
import numpy as np
import pandas as pd
import os
import shutil
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D , MaxPool2D , Flatten , Dropout , BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import VGG16,DenseNet121
from keras.callbacks import EarlyStopping
from utils import get_data # Assurez-vous d'avoir le fichier utils.py dans content

### Import des données

In [ ]:
# Installer le drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Installer kaggle
!pip install kaggle

In [ ]:
# Définir kaggle configuration
import os
os.environ['KAGGLE_CONFIG_DIR'] = '/content/drive/MyDrive/kaggle'
# Mettre le fichier kaggle.json dans le dossier kaggle

In [ ]:
# Télécharger les données
!kaggle datasets download -d tawsifurrahman/tuberculosis-tb-chest-xray-dataset

In [ ]:
# Déziper le fichier des données
!unzip /content/tuberculosis-tb-chest-xray-dataset.zip

In [ ]:
# Récupérer les données images converties en matrice et labelisées en utilisant la fonction get_data
Normal = get_data('/content/TB_Chest_Radiography_Database/Normal', 0)
Tuberculosis = get_data('/content/TB_Chest_Radiography_Database/Tuberculosis',1)

In [ ]:
# Afficher l'image en matrice de la première image normal et son label
Normal[1]

### Exploration et preparation des données

In [ ]:
# Images normal et labels
normal_images = [image[0] for image in Normal]
normal_labels = [label[1] for label in Normal]
# Images tuberculosis et labels
tuberculosis_images = [image[0] for image in Tuberculosis]
tuberculosis_labels = [label[1] for label in Tuberculosis]

In [ ]:
# Matrice des images
data_images = np.array(normal_images + tuberculosis_images)
# Labels des images
data_labels = np.array(normal_labels + tuberculosis_labels)

In [ ]:
# Afficher la dimension
data_images.shape

In [ ]:
# Visualiser les 25 premières images des données
labels_str = ['Normal', 'Tuberculosis']
plt.figure(figsize=(10,10))
for i in range(25):
  plt.subplot(5,5,i+1)
  plt.xticks([])
  plt.yticks([])
  plt.imshow(data_images[data_images.shape[0] - i - 1], cmap = 'gray')
  plt.title(labels_str[data_labels[data_images.shape[0] - i - 1]])

In [ ]:
# Nombre d'images  dans chaque classe
pd.Series(data_labels).value_counts()

In [ ]:
# Barplot du nombre d'images dans chaque classe
labels = ['Normal', 'Tuberculosis']
labels_numb = pd.Series(data_labels).value_counts().values
sns.barplot(x = labels, y =labels_numb, hue =labels,  palette= sns.husl_palette(2))

In [ ]:
# spliter les données
from sklearn.model_selection import train_test_split
train_images,test_images, train_labels,test_labels = train_test_split(data_images, data_labels, test_size= 0.2, random_state=42)

In [ ]:
# Normaliser (en float32 pour économiser la mémoire)
train_images,test_images = (train_images/255).astype('float32'), (test_images/255).astype('float32')

In [ ]:
# Ajouter la dimension des canaux : (N, 150, 150) -> (N, 150, 150, 1)
train_images = np.expand_dims(train_images, axis=-1)
test_images = np.expand_dims(test_images, axis=-1)

In [ ]:
# Répéter le canal gris 3 fois : (N, 150, 150, 1) -> (N, 150, 150, 3)
train_images = np.repeat(train_images, 3, axis=-1)
test_images = np.repeat(test_images, 3, axis=-1)

### Fonctions d'évaluation

Le jeu de données est déséquilibré (beaucoup plus d'images `Normal` que de `Tuberculosis`) : en plus de l'accuracy, on suit la **precision** et le **recall**, et on calcule le **F1-score** pour comparer les modèles.

In [ ]:
# Importer ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# Fonction affichant la matrice de confusion d'un modèle sur les données validation
def CM_visualisation(model_objet):
    # Prédictions du modèle : probabilité > 0.5 => Tuberculosis (1), sinon Normal (0)
    test_pred = (model_objet.predict(test_images, verbose = 0) > 0.5).astype(int).ravel()
    # Confusion matrix
    cnf_matr = confusion_matrix(test_labels,test_pred, labels=[0,1])

    # Utiliser ConfusionMatrixDisplay pour visualiser la matrice de confusion
    disp = ConfusionMatrixDisplay(confusion_matrix=cnf_matr,
                                  display_labels=['Normal','Tuberculosis'])
    disp.plot() # Afficher la matrice de confusion
    plt.title('Matrice de confusion') # Définir le titre
    plt.show() # Afficher la figure

# Fonction affichant les courbes d'accuracy et de loss (train et validation)
def courbes(history_objet):
    # Evolution de l'accuracy sur les données d'entrainement et de validation suivant les différents epochs
    plt.plot(history_objet.history['accuracy'])
    plt.plot(history_objet.history['val_accuracy'])
    plt.title("Evolution l'accuracy sur train et val data")
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend(['Train', 'Validation'], loc = 'lower right')
    plt.show()
    # Evolution du loss sur les données d'entrainement et de validation suivant les différents epochs
    plt.plot(history_objet.history['loss'])
    plt.plot(history_objet.history['val_loss'])
    plt.title("Evolution de loss sur train et val data")
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend(['Train', 'Validation'], loc = 'upper right')
    plt.show()

# Fonction retournant loss, accuracy, precision, recall et F1-score d'un modèle sur des données
def metriques(model_objet, images, labels):
    res = model_objet.evaluate(images, labels, verbose = 0)
    f1 = 2 * res[2] * res[3] / (res[2] + res[3]) if (res[2] + res[3]) > 0 else 0
    return {'Loss': res[0], 'Accuracy': res[1], 'Precision': res[2], 'Recall': res[3], 'F1-score': f1}

# Dictionnaires qui vont contenir les modèles finaux et leurs résultats
models = {}
resultats_val = {}
resultats_train = {}

### Installer et importer Keras Tuner

Pour chaque modèle : recherche des hyperparamètres optimaux (Hyperband, objectif `val_accuracy`), puis réentraînement du modèle final avec ces valeurs. Le modèle est défini dans une fonction `build_model_...(hp)` ; le modèle final est construit avec cette même fonction et les hyperparamètres optimaux (`best_hps`), ce qui garantit que l'architecture finale est celle trouvée par le tuner. Comme dans le cours, le jeu `test` sert de données de validation.

In [ ]:
# Installer keras tuner
!pip install keras-tuner --upgrade

In [ ]:
# importer le package keras_tuner
import keras_tuner as kt

### Modèle de base

Hyperparamètres recherchés : filtres des 4 couches convolutives (16, 32 ou 64), neurones des 3 couches Dense (32, 64 ou 128) et learning rate d'Adam (1e-2, 1e-3 ou 1e-4).

In [ ]:
# fonction build_model du modèle de base
def build_model_baseline(hp):
    # Choisir les hyperparamètres
    hp_filters1 = hp.Choice('filters1', values = [16, 32, 64])
    hp_filters2 = hp.Choice('filters2', values = [16, 32, 64])
    hp_filters3 = hp.Choice('filters3', values = [16, 32, 64])
    hp_filters4 = hp.Choice('filters4', values = [16, 32, 64])
    hp_dense1 = hp.Choice('dense1', values = [32, 64, 128])
    hp_dense2 = hp.Choice('dense2', values = [32, 64, 128])
    hp_dense3 = hp.Choice('dense3', values = [32, 64, 128])
    hp_lr = hp.Choice('learning_rate', values = [1e-2, 1e-3, 1e-4])

    model = Sequential() # initialiser l'architecture CNN
    # Couches convolutives et pooling
    model.add(Conv2D(hp_filters1, (3, 3), activation='relu', padding = 'same', input_shape=(150,150,3)))
    model.add(MaxPool2D((2, 2)))
    model.add(Conv2D(hp_filters2, (3, 3), activation='relu', padding = 'same'))
    model.add(MaxPool2D((2, 2)))
    model.add(Conv2D(hp_filters3, (3, 3), activation='relu', padding = 'same'))
    model.add(MaxPool2D((2, 2)))
    model.add(Conv2D(hp_filters4, (3, 3), activation='relu', padding = 'same'))
    model.add(MaxPool2D((2, 2)))
    # couches fully connected
    model.add(Flatten()) # couche flatten
    model.add(Dense(units = hp_dense1, activation = 'relu')) # Dense 1
    model.add(Dense(units = hp_dense2, activation = 'relu')) # Dense 2
    model.add(Dense(units = hp_dense3, activation = 'relu')) # Dense 3
    model.add(Dense(units = 1, activation = 'sigmoid')) # sortie binaire

    # compiler le modèle avec le learning rate choisi
    model.compile(optimizer = Adam(learning_rate = hp_lr), loss = 'binary_crossentropy', metrics = ['accuracy', 'precision', 'recall'])
    return model

In [ ]:
# Initialiser le tuner du modèle de base
tuner_baseline = kt.Hyperband(build_model_baseline,
                     objective = 'val_accuracy',
                     max_epochs = 10,
                     directory = 'tuner_dir',
                     project_name = 'baseline_tb',
                     overwrite = True)

# lancer la recherche avec le tuner
tuner_baseline.search(train_images, train_labels, validation_data = (test_images, test_labels), epochs = 10, batch_size = 64)

In [ ]:
# Afficher les paramètres optimaux du modèle de base
best_hps_baseline = tuner_baseline.get_best_hyperparameters()[0]
for h_param in ['filters1', 'filters2', 'filters3', 'filters4', 'dense1', 'dense2', 'dense3', 'learning_rate']:
    print(h_param, best_hps_baseline.get(h_param))

In [ ]:
# Créer le modèle de base final avec les hyperparamètres optimaux
model = build_model_baseline(best_hps_baseline)
# summary
model.summary()

In [ ]:
# Définir le early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=3)
# Entrainer le modèle (compilé dans build_model_baseline : Adam, binary_crossentropy, accuracy, precision, recall)
history = model.fit(train_images, train_labels, validation_data =(test_images,test_labels), epochs=20,batch_size= 64, callbacks = early_stopping)
models['Baseline'] = model

#### Evaluation

In [ ]:
# matrice de confusion
CM_visualisation(model)

In [ ]:
# Afficher les courbes d'accuracy et de loss
courbes(history)

In [ ]:
# Accuracy,loss,precision et recall sur validation data et sur train data
resultats_val['Baseline'] = metriques(model, test_images, test_labels)
resultats_train['Baseline'] = metriques(model, train_images, train_labels)
print('Validation :', resultats_val['Baseline'])
print('Train :', resultats_train['Baseline'])

### Modèle VGG16

Base `imagenet` gelée. Hyperparamètres recherchés : Dense 1 (256, 512 ou 1024), Dense 2 (128, 256 ou 512), Dropout 1 (0.3 à 0.6), Dropout 2 (0.2 à 0.5) et learning rate (1e-3 ou 1e-4).

In [ ]:
# fonction build_model de VGG16
def build_model_vgg16(hp):
    # Choisir les hyperparamètres
    hp_dense1 = hp.Choice('dense1', values = [256, 512, 1024])
    hp_dense2 = hp.Choice('dense2', values = [128, 256, 512])
    hp_dropout1 = hp.Choice('dropout1', values = [0.3, 0.4, 0.5, 0.6])
    hp_dropout2 = hp.Choice('dropout2', values = [0.2, 0.3, 0.4, 0.5])
    hp_lr = hp.Choice('learning_rate', values = [1e-3, 1e-4])

    # Charger VGG16 préentrainé sans les couches de classification (Sans couches fully connected)
    base_vgg16 = VGG16(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
    # Geler les couches de base (Leurs poids, appris sur ImageNet, restent fixes pendant l'entraînement)
    base_vgg16.trainable = False

    # Construire le modèle VGG16 avec Sequential
    vgg16_model = Sequential()
    vgg16_model.add(base_vgg16) # ajouter les couches de convolutions du modèle vgg16
    vgg16_model.add(Flatten()) # Applanir
    vgg16_model.add(Dense(units = hp_dense1, activation='relu')) # Couche Dense 1
    vgg16_model.add(Dropout(hp_dropout1)) # Couche Dropout 1
    vgg16_model.add(Dense(units = hp_dense2, activation='relu')) # Couche Dense 2
    vgg16_model.add(Dropout(hp_dropout2)) # Couche Dropout 2
    vgg16_model.add(Dense(units = 1, activation='sigmoid')) # Couche de sortie pour classification binaire

    # compiler le modèle avec le learning rate choisi
    vgg16_model.compile(optimizer = Adam(learning_rate = hp_lr), loss = 'binary_crossentropy', metrics = ['accuracy', 'precision', 'recall'])
    return vgg16_model

In [ ]:
# Initialiser le tuner de VGG16
tuner_vgg16 = kt.Hyperband(build_model_vgg16,
                     objective = 'val_accuracy',
                     max_epochs = 10,
                     directory = 'tuner_dir',
                     project_name = 'vgg16_tb',
                     overwrite = True)

# lancer la recherche avec le tuner
tuner_vgg16.search(train_images, train_labels, validation_data = (test_images, test_labels), epochs = 10, batch_size = 64)

In [ ]:
# Afficher les paramètres optimaux de VGG16
best_hps_vgg16 = tuner_vgg16.get_best_hyperparameters()[0]
for h_param in ['dense1', 'dense2', 'dropout1', 'dropout2', 'learning_rate']:
    print(h_param, best_hps_vgg16.get(h_param))

In [ ]:
# Créer le modèle VGG16 final avec les hyperparamètres optimaux
vgg16_model = build_model_vgg16(best_hps_vgg16)
# summary
vgg16_model.summary()

In [ ]:
# Entraînement VGG16 (compilé dans build_model_vgg16 : Adam, binary_crossentropy, accuracy, precision, recall)
history_vgg16 = vgg16_model.fit(train_images, train_labels, validation_data= (test_images, test_labels), epochs = 20,batch_size = 64, callbacks=early_stopping)
models['VGG16'] = vgg16_model

#### Evaluation

In [ ]:
# matrice de confusion
CM_visualisation(vgg16_model)

In [ ]:
# Afficher les courbes d'accuracy et de loss
courbes(history_vgg16)

In [ ]:
# Accuracy,loss,precision et recall sur validation data et sur train data
resultats_val['VGG16'] = metriques(vgg16_model, test_images, test_labels)
resultats_train['VGG16'] = metriques(vgg16_model, train_images, train_labels)
print('Validation :', resultats_val['VGG16'])
print('Train :', resultats_train['VGG16'])

### Modèle DenseNet121

Mêmes hyperparamètres que VGG16, avec la base `DenseNet121` gelée.

In [ ]:
# fonction build_model de DenseNet121
def build_model_densenet121(hp):
    # Choisir les hyperparamètres
    hp_dense1 = hp.Choice('dense1', values = [256, 512, 1024])
    hp_dense2 = hp.Choice('dense2', values = [128, 256, 512])
    hp_dropout1 = hp.Choice('dropout1', values = [0.3, 0.4, 0.5, 0.6])
    hp_dropout2 = hp.Choice('dropout2', values = [0.2, 0.3, 0.4, 0.5])
    hp_lr = hp.Choice('learning_rate', values = [1e-3, 1e-4])

    # Charger DenseNet121 préentrainé sans les couches de classification (Sans couches fully connected)
    base_densenet = DenseNet121(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
    # Geler les couches de base
    base_densenet.trainable = False

    # Construire le modèle DenseNet121 avec Sequential
    densenet_model = Sequential()
    densenet_model.add(base_densenet) # ajouter les couches de convolutions du modèle DenseNet121
    densenet_model.add(Flatten()) # Applanir
    densenet_model.add(Dense(units = hp_dense1, activation='relu')) # Couche Dense 1
    densenet_model.add(Dropout(hp_dropout1)) # Couche Dropout 1
    densenet_model.add(Dense(units = hp_dense2, activation='relu')) # Couche Dense 2
    densenet_model.add(Dropout(hp_dropout2)) # Couche Dropout 2
    densenet_model.add(Dense(units = 1, activation='sigmoid')) # Couche de sortie pour classification binaire

    # compiler le modèle avec le learning rate choisi
    densenet_model.compile(optimizer = Adam(learning_rate = hp_lr), loss = 'binary_crossentropy', metrics = ['accuracy', 'precision', 'recall'])
    return densenet_model

In [ ]:
# Initialiser le tuner de DenseNet121
tuner_densenet = kt.Hyperband(build_model_densenet121,
                     objective = 'val_accuracy',
                     max_epochs = 10,
                     directory = 'tuner_dir',
                     project_name = 'densenet121_tb',
                     overwrite = True)

# lancer la recherche avec le tuner
tuner_densenet.search(train_images, train_labels, validation_data = (test_images, test_labels), epochs = 10, batch_size = 64)

In [ ]:
# Afficher les paramètres optimaux de DenseNet121
best_hps_densenet = tuner_densenet.get_best_hyperparameters()[0]
for h_param in ['dense1', 'dense2', 'dropout1', 'dropout2', 'learning_rate']:
    print(h_param, best_hps_densenet.get(h_param))

In [ ]:
# Créer le modèle DenseNet121 final avec les hyperparamètres optimaux
densenet_model = build_model_densenet121(best_hps_densenet)
# summary
densenet_model.summary()

In [ ]:
# Entraînement DenseNet121 (compilé dans build_model_densenet121 : Adam, binary_crossentropy, accuracy, precision, recall)
history_densenet = densenet_model.fit(train_images, train_labels, validation_data = (test_images,test_labels), epochs = 20,batch_size =64, callbacks=early_stopping)
models['DenseNet121'] = densenet_model

#### Evaluation

In [ ]:
# Matrice de confusion
CM_visualisation(densenet_model)

In [ ]:
# Afficher les courbes d'accuracy et de loss
courbes(history_densenet)

In [ ]:
# Accuracy,loss,precision et recall sur validation data et sur train data
resultats_val['DenseNet121'] = metriques(densenet_model, test_images, test_labels)
resultats_train['DenseNet121'] = metriques(densenet_model, train_images, train_labels)
print('Validation :', resultats_val['DenseNet121'])
print('Train :', resultats_train['DenseNet121'])

### Comparaison des trois modèles

In [ ]:
# Tableau comparatif des métriques sur les données de validation
comparaison_val = pd.DataFrame(resultats_val).T
comparaison_val

In [ ]:
# Tableau comparatif des métriques sur les données d'entrainement
comparaison_train = pd.DataFrame(resultats_train).T
comparaison_train

### Télécharger le modèle le plus performant

Le jeu de données étant déséquilibré, le meilleur modèle est choisi avec le **F1-score** sur les données de validation (compromis entre precision et recall), et non avec l'accuracy seule. On vérifie ainsi si c'est toujours VGG16 après la recherche d'hyperparamètres.

In [ ]:
# Choisir et sauvegarder le meilleur modèle
best_name = comparaison_val['F1-score'].idxmax()
best_model = models[best_name]
print('Meilleur modèle :', best_name)
best_model.save('tuberculosis_best_model.h5')
best_model.save('tuberculosis_best_model.keras')

In [ ]:
# Copier le meilleur modèle dans le drive (dossier du projet)
dossier = '/content/drive/MyDrive/Master Bambey-Deep Learning /Projet/TP5'
os.makedirs(dossier, exist_ok = True)
shutil.copy('tuberculosis_best_model.h5', dossier)
shutil.copy('tuberculosis_best_model.keras', dossier)
os.listdir(dossier)

### Prédictions

La fonction de prétraitement reproduit **exactement** celui de l'entraînement (`get_data` : image lue en niveaux de gris, redimensionnée en 150×150, divisée par 255, canal répété 3 fois). Elle sera reprise telle quelle dans `app.py`.

In [ ]:
# Fonction convertissant l'image à la dimension exacte utilisée sur le training
import cv2
from PIL import Image

def imageToArray(image):
  width, height = 150, 150  # dimensions requises par notre modèle

  image = np.array(image.convert('L'))        # niveaux de gris (comme cv2.IMREAD_GRAYSCALE dans get_data)
  image = cv2.resize(image, (width, height))  # même redimensionnement que get_data
  image_array = image / 255.0                 # Normaliser les valeurs des pixels entre 0 et 1
  image_array = np.repeat(image_array[..., np.newaxis], 3, axis=-1)  # répéter le canal gris 3 fois

  # Redimensionner la matrice image pour qu'elle correspond à la forme d'entrée de notre modèle
  return image_array.reshape(-1, width, height, 3).astype('float32')

In [ ]:
# Predictions avec le meilleur modèle
class_names = ['Normal', 'Tuberculosis']
image_path = '/content/drive/MyDrive/Master Bambey-Deep Learning /DL 5/images_test/Tuberculosis_test1.png'
image = Image.open(image_path)
proba = float(best_model.predict(imageToArray(image), verbose = 0)[0][0])
pred = class_names[int(np.round(proba))]
plt.imshow(image) # afficher l'image
plt.xlabel(f'Prédiction : {pred} (probabilité de tuberculose : {proba:.1%})')
plt.show()

### Préparation pour le déploiement avec Streamlit

In [ ]:
# Fonction de prédiction retournant la probabilité de chaque classe (même prétraitement que l'application Streamlit)
from tensorflow.keras.models import load_model
model_app = load_model('tuberculosis_best_model.h5')

def predict(image):
    p_tb = float(model_app.predict(imageToArray(image), verbose = 0)[0][0])
    return {class_names[1]: p_tb, class_names[0]: 1.0 - p_tb}

# Tester la fonction sur une image
predict(Image.open(image_path))